# Task 2 — Real Estate Housing Price Prediction
## Ridge & Lasso Regression | Datheon Machine Learning & Data Science Internship

**Objective.** Predict median house values using the California Housing dataset and compare Ordinary Least Squares (OLS), Ridge (L2), and Lasso (L1) regression with RMSE, MAE, and R².

**Workflow:** business framing → data validation → EDA → feature engineering → train/test split → preprocessing pipeline → cross-validated hyperparameter tuning → hold-out evaluation → residual analysis → conclusions.

> **Reproducibility note:** The dataset is loaded using scikit-learn's dataset loader and may require internet access on the first run. Metrics are calculated by the notebook; no results are fabricated or hard-coded.

## 1. Business question and success criteria

A housing valuation model can help analysts estimate relative property values from area-level attributes. This dataset contains California census-block-group characteristics, so the target is a **block-group median house value**, not an individual home's appraisal.

We will compare:
- **OLS / Linear Regression:** baseline linear model.
- **Ridge:** L2 regularization, which shrinks coefficients and can stabilize correlated predictors.
- **Lasso:** L1 regularization, which can shrink some coefficients to zero.

**Primary comparison metric:** RMSE (penalizes large errors). Also report MAE (typical absolute error) and R² (variance explained). Use the test set only for final evaluation.

In [ ]:
# If needed in a fresh environment, run:
# %pip install -r requirements.txt

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

## 2. Load and inspect the dataset

The target `MedHouseVal` is expressed in units of $100,000. For example, a target of 2.5 corresponds to a median value of approximately $250,000 in the dataset's original scale.

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()
df = df.rename(columns={"MedHouseVal": "MedHouseVal_100k"})

print("Shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))
display(df.describe().T)

## 3. Data quality checks

Check duplicates, missing values, and the target distribution before modeling. We retain the official target definition and avoid using any information from the test set to fit preprocessing steps.

In [ ]:
print("Duplicate rows:", int(df.duplicated().sum()))
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_count"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["MedHouseVal_100k"], bins=40, kde=True, ax=axes[0])
axes[0].set_title("Target distribution")
axes[0].set_xlabel("Median house value (units of $100,000)")
sns.boxplot(x=df["MedHouseVal_100k"], ax=axes[1])
axes[1].set_title("Target outliers / upper cap")
plt.tight_layout()
plt.show()

## 4. Exploratory analysis and multicollinearity

Correlation is a diagnostic, not proof of causation. Highly correlated predictors can make OLS coefficients unstable; Ridge can help stabilize them. Lasso may select a sparse subset of features.

In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap="vlag", center=0, annot=True, fmt=".2f", square=True)
plt.title("Feature and target correlation matrix")
plt.tight_layout()
plt.show()

feature_corr = corr.drop(index="MedHouseVal_100k", columns="MedHouseVal_100k")
pairs = []
cols = feature_corr.columns.tolist()
for i, a in enumerate(cols):
    for b in cols[i+1:]:
        value = feature_corr.loc[a, b]
        if abs(value) >= 0.70:
            pairs.append((a, b, value))
print("Feature pairs with |correlation| >= 0.70:")
display(pd.DataFrame(pairs, columns=["feature_1", "feature_2", "correlation"]))

## 5. Feature engineering and leakage-safe split

We add a small, interpretable set of interaction features using `PolynomialFeatures(interaction_only=True)`. This creates pairwise interactions without adding squared terms. All preprocessing is placed inside the pipeline so imputation, scaling, and feature generation are fit on training folds only.

The data are randomly split 80/20. Since observations are geographic, this random split is useful for the assignment benchmark but does **not** guarantee performance on a completely new region or future period.

In [ ]:
X = df.drop(columns="MedHouseVal_100k")
y = df["MedHouseVal_100k"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print(f"Train rows: {len(X_train):,} | Test rows: {len(X_test):,}")
print("Features:", X.columns.tolist())

## 6. Build pipelines and tune regularization with GridSearchCV

Scaling is important for Ridge and Lasso because the penalty depends on coefficient magnitude. The same preprocessing is applied consistently across models. Five-fold cross-validation selects alpha using negative RMSE; the held-out test set is not used for tuning.

In [ ]:
def make_pipeline(model):
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("interactions", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
        ("scaler", StandardScaler()),
        ("model", model),
    ])

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

searches = {
    "OLS": GridSearchCV(
        make_pipeline(LinearRegression()),
        param_grid={"model__fit_intercept": [True]},
        scoring="neg_root_mean_squared_error", cv=cv, n_jobs=-1
    ),
    "Ridge": GridSearchCV(
        make_pipeline(Ridge()),
        param_grid={"model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
        scoring="neg_root_mean_squared_error", cv=cv, n_jobs=-1
    ),
    "Lasso": GridSearchCV(
        make_pipeline(Lasso(max_iter=20000)),
        param_grid={"model__alpha": [0.0001, 0.001, 0.01, 0.1, 1.0]},
        scoring="neg_root_mean_squared_error", cv=cv, n_jobs=-1
    ),
}

for name, search in searches.items():
    print(f"Fitting {name}...")
    search.fit(X_train, y_train)
    print(f"  Best CV RMSE: {-search.best_score_:.4f}; params: {search.best_params_}")

## 7. Evaluate on the untouched test set

The target is in units of $100,000, so multiply MAE/RMSE by 100,000 for an approximate dollar interpretation. These are dataset-level estimates and should not be represented as individual property appraisals.

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "RMSE (target units)": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE (target units)": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
        "RMSE (approx. USD)": float(np.sqrt(mean_squared_error(y_true, y_pred)) * 100_000),
        "MAE (approx. USD)": float(mean_absolute_error(y_true, y_pred) * 100_000),
    }

rows = []
predictions = {}
for name, search in searches.items():
    pred = search.best_estimator_.predict(X_test)
    predictions[name] = pred
    rows.append({"Model": name, **regression_metrics(y_test, pred),
                 "Best parameters": str(search.best_params_)})
results = pd.DataFrame(rows).sort_values("RMSE (target units)")
display(results.round(4))
best_name = results.iloc[0]["Model"]
print("Lowest test RMSE in this run:", best_name)

## 8. Predicted vs actual and residual diagnostics

A good model should place points near the 45-degree line. Residual plots help identify systematic errors, skew, and non-constant variance. They do not alone establish that model assumptions are satisfied.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, (name, pred) in zip(axes, predictions.items()):
    ax.scatter(y_test, pred, s=8, alpha=0.25)
    low = min(float(y_test.min()), float(np.min(pred)))
    high = max(float(y_test.max()), float(np.max(pred)))
    ax.plot([low, high], [low, high], linestyle="--")
    ax.set_title(f"{name}: actual vs predicted")
    ax.set_xlabel("Actual target")
    ax.set_ylabel("Predicted target")
plt.tight_layout()
plt.show()

best_pred = predictions[best_name]
residuals = y_test.to_numpy() - best_pred
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(best_pred, residuals, s=9, alpha=0.3)
axes[0].axhline(0, linestyle="--")
axes[0].set_xlabel("Predicted target")
axes[0].set_ylabel("Residual (actual - predicted)")
axes[0].set_title(f"Residuals vs predictions — {best_name}")
sns.histplot(residuals, bins=40, kde=True, ax=axes[1])
axes[1].set_title("Residual distribution")
axes[1].set_xlabel("Residual")
plt.tight_layout()
plt.show()

## 9. Model interpretation and limitations

For Ridge/Lasso, coefficients correspond to the transformed feature space (after interaction generation and scaling). The transformed feature names can be inspected, but coefficient magnitude should not be interpreted as a causal effect.

Limitations:
- The dataset is old and represents aggregate census-block-group statistics.
- Target values have an upper cap in the source data.
- Random train/test splitting can overstate generalization to geographically separate areas.
- Housing markets change; a production model would require current data, drift monitoring, fairness review, and domain validation.

In [ ]:
best_estimator = searches[best_name].best_estimator_
poly = best_estimator.named_steps["interactions"]
feature_names = poly.get_feature_names_out(X.columns)
model = best_estimator.named_steps["model"]

if hasattr(model, "coef_"):
    coef_table = pd.DataFrame({
        "feature": feature_names,
        "coefficient": np.asarray(model.coef_).ravel()
    })
    coef_table["abs_coefficient"] = coef_table["coefficient"].abs()
    display(coef_table.sort_values("abs_coefficient", ascending=False).head(15))
else:
    print("This estimator does not expose linear coefficients.")

## 10. Conclusion template — complete after running

Use the actual metric table above to fill in the conclusion; do not claim a model is best unless the results support it.

- Best test RMSE model: **[fill from results]**
- Test RMSE / MAE / R²: **[copy values from results table]**
- Did regularization improve on OLS? **[compare the measured results]**
- Most important limitation: the target is a regional median value, not an individual house appraisal.
- Next experiment: compare a tree-based regressor and test a geographically separated validation split.

## Submission checklist
- [ ] Run all cells from top to bottom in a clean environment.
- [ ] Check that the metric table and plots render.
- [ ] Save the executed notebook with outputs.
- [ ] Add a short README entry describing the dataset, approach, metrics, and limitations.
- [ ] Push the notebook to your public GitHub repository.